# Day 3 – Tool Calling Basics with LangChain

## Objectives

- Understand Tool Calling
- Learn how Large Language Models use external tools
- Create custom tools using LangChain
- Build a multi-tool AI agent
- Test individual tools
- Bind tools to a Large Language Model
- Understand tool routing and execution

# What is Tool Calling?

Tool Calling allows a Large Language Model (LLM) to interact with external functions instead of relying only on its internal knowledge.

Instead of generating every answer directly, the model can decide when to call a tool such as:

- Calculator
- Weather API
- Database Search
- Currency Converter
- File Reader

This enables AI systems to perform real-world tasks accurately and efficiently.

# Why Tool Calling?

Without tool calling, an LLM is limited to the knowledge available in its training data.

Tool calling enables an AI system to:

- Perform mathematical calculations
- Retrieve live information
- Search databases
- Execute custom Python functions
- Access external APIs
- Automate real-world workflows

# Tool Calling Workflow

User Query

        │
        ▼

Large Language Model

        │
        ▼

Select Appropriate Tool

        │
        ▼

Execute Tool

        │
        ▼

Return Tool Output

        │
        ▼
Generate Final Response

In [1]:
from dotenv import load_dotenv
import os

from langchain_groq import ChatGroq
from langchain_core.tools import tool

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM Initialized Successfully!")

Groq LLM Initialized Successfully!


# Creating Our First Tool

LangChain provides the `@tool` decorator to convert a normal Python function into a tool that can be called by an LLM.

Our first tool performs simple arithmetic calculations.

In [2]:
@tool
def calculate(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [3]:
print(calculate.invoke("25 * 8"))
print(calculate.invoke("(45 + 15) / 3"))

200
20.0


# Tool 2 – Weather Lookup

In a real-world application, this tool would call a weather API to retrieve live weather information.

For this notebook, we will simulate weather data to understand how tool calling works without requiring an external API.

In [6]:
@tool
def weather(city: str) -> str:
    """
    Returns the weather for a given city.
    """

    weather_data = {
        "Islamabad": "Sunny, 42°C",
        "Lahore": "Cloudy, 45°C",
        "Karachi": "Hot, 47°C",
        "Peshawar": "Sunny, 44°C"
    }

    return weather_data.get(city, "Weather data not available.")

In [7]:
print(weather.invoke("Islamabad"))
print(weather.invoke("Lahore"))
print(weather.invoke("Karachi"))

Sunny, 42°C
Cloudy, 45°C
Hot, 47°C


# Tool 3 – Database Search

Many AI systems need to retrieve information from an internal database.

For demonstration purposes, we will use a Python dictionary as a mock database.

In [8]:
@tool
def search_db(topic: str) -> str:
    """
    Searches a simple database.
    """

    database = {
        "python": "Python is a high-level programming language.",
        "langchain": "LangChain is a framework for LLM applications.",
        "groq": "Groq provides fast inference for Large Language Models.",
        "pydantic": "Pydantic validates structured data using Python type hints."
    }

    return database.get(topic.lower(), "No information found.")

In [9]:
print(search_db.invoke("python"))
print(search_db.invoke("langchain"))
print(search_db.invoke("groq"))

Python is a high-level programming language.
LangChain is a framework for LLM applications.
Groq provides fast inference for Large Language Models.


# Tool 4 – Date Formatter

Formatting dates into a consistent style is a common task in AI applications.

In [10]:
from datetime import datetime

@tool
def format_date(date_string: str) -> str:
    """
    Converts YYYY-MM-DD into a readable format.
    """

    try:
        date = datetime.strptime(date_string, "%Y-%m-%d")
        return date.strftime("%d %B %Y")

    except Exception:
        return "Invalid date format."

In [11]:
print(format_date.invoke("2026-07-02"))
print(format_date.invoke("2025-12-06"))

02 July 2026
06 December 2025


# Tool 5 – Currency Converter

This tool demonstrates a simple currency conversion using predefined exchange rates.

In production systems, exchange rates would typically be fetched from a live financial API.

In [12]:
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Converts currency using fixed exchange rates.
    """

    rates = {
        ("USD", "PKR"): 285,
        ("PKR", "USD"): 1 / 285,
        ("USD", "EUR"): 0.85,
        ("EUR", "USD"): 1.18,
    }

    key = (from_currency.upper(), to_currency.upper())

    if key not in rates:
        return "Conversion not supported."

    converted = amount * rates[key]

    return f"{converted:.2f} {to_currency.upper()}"

In [13]:
print(convert_currency.invoke({
    "amount": 100,
    "from_currency": "USD",
    "to_currency": "PKR"
}))

print(convert_currency.invoke({
    "amount": 57000,
    "from_currency": "PKR",
    "to_currency": "USD"
}))

28500.00 PKR
200.00 USD


# Tool 6 – Text Summarizer

Summarization is a common AI task that condenses long pieces of text into shorter, meaningful summaries.

For this example, we will implement a simple summarizer that returns the first sentence of the provided text.

In [14]:
@tool
def summarize_text(text: str) -> str:
    """
    Returns a simple summary by extracting the first sentence.
    """

    sentences = text.split(".")

    if sentences and sentences[0].strip():
        return sentences[0].strip() + "."

    return "No summary available."

In [15]:
sample_text = """
LangChain is a framework for developing applications powered by Large Language Models.
It simplifies prompt engineering, tool calling, memory management, and agent development.
"""

print(summarize_text.invoke(sample_text))

LangChain is a framework for developing applications powered by Large Language Models.


# Binding Tools to the LLM

LangChain allows us to bind multiple tools to an LLM using the `bind_tools()` method.

This enables the model to decide which tool should be used based on the user's request.

In [16]:
tools = [
    calculate,
    weather,
    search_db,
    format_date,
    convert_currency,
    summarize_text
]

llm_with_tools = llm.bind_tools(tools)

print("All tools successfully bound to the LLM!")

All tools successfully bound to the LLM!


# Testing Tool Calling

When given a user query, the LLM can determine whether a tool should be used and identify the appropriate tool to call.

In [19]:
response = llm_with_tools.invoke("What is 45 * 8?")

print("LLM Response:")
print(response)

print("\nTool Calls:")
print(response.tool_calls)

LLM Response:
content='' additional_kwargs={'tool_calls': [{'id': '0p821329d', 'function': {'arguments': '{"expression":"45 * 8"}', 'name': 'calculate'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 511, 'total_tokens': 528, 'completion_time': 0.037049591, 'completion_tokens_details': None, 'prompt_time': 0.028334302, 'prompt_tokens_details': None, 'queue_time': 0.013575363, 'total_time': 0.065383893}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f218a-3e4f-7b90-834a-0a8b12e5a472-0' tool_calls=[{'name': 'calculate', 'args': {'expression': '45 * 8'}, 'id': '0p821329d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 511, 'output_tokens': 17, 'total_tokens': 528}

Tool Calls:
[{'name': 'calculate', 'args': {'expression': '45 * 8'}, 'id': '0p821329d', 'type

# Individual Tool Demonstration

Each custom tool created in this notebook has been tested independently to verify its functionality.

These tools can now be integrated into larger AI agent workflows.

In [18]:
print("Calculator:", calculate.invoke("12 + 18"))

print("Weather:", weather.invoke("Islamabad"))

print("Database:", search_db.invoke("langchain"))

print("Formatted Date:", format_date.invoke("2026-07-02"))

print(
    "Currency:",
    convert_currency.invoke({
        "amount": 100,
        "from_currency": "USD",
        "to_currency": "PKR"
    })
)

print(
    "Summary:",
    summarize_text.invoke(
        "Artificial Intelligence is transforming many industries. It enables automation and intelligent decision-making."
    )
)

Calculator: 30
Weather: Sunny, 42°C
Database: LangChain is a framework for LLM applications.
Formatted Date: 02 July 2026
Currency: 28500.00 PKR
Summary: Artificial Intelligence is transforming many industries.


# Observations

During this notebook, the following observations were made:

- LangChain makes it easy to convert Python functions into tools.
- Tool calling extends the capabilities of Large Language Models.
- Individual tools can be developed and tested independently.
- Multiple tools can be bound to a single LLM.
- Tool calling forms the foundation for AI agents capable of solving complex tasks.

# Conclusion

In this notebook, I explored the fundamentals of tool calling using LangChain and Groq.

I created multiple custom tools, tested each one individually, and bound them to a Large Language Model. This demonstrates how LLMs can interact with external functions to perform calculations, retrieve information, format data, and process text.

These concepts provide the foundation for building more advanced AI agents, which will be explored in Lab 2.2.